第 1 步：先做一个最小 PyTorch 模型

In [ ]:
import torch
import torch.nn as nn

class MiniModel(nn.Module):
    def forward(self, x):
        y = torch.relu(x)
        z = y + 1.0
        return z

model = MiniModel().eval()
x = torch.tensor([[-1.0, 0.5, 2.0]], dtype=torch.float32)

torch.onnx.export(
    model,
    x,
    "work/mini.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=13
)

print("exported: work/mini.onnx")

把 ONNX 里的 Add 改成 custom op

In [2]:
import onnx
from onnx import helper

model = onnx.load("work/mini.onnx")
graph = model.graph

new_nodes = []
for node in graph.node:
    if node.op_type == "Add":
        custom_node = helper.make_node(
            "MyScale",
            inputs=[node.input[0]],
            outputs=list(node.output),
            domain="my.custom",
            alpha=2.0
        )
        new_nodes.append(custom_node)
    else:
        new_nodes.append(node)

del graph.node[:]
graph.node.extend(new_nodes)

# 清理未被引用的 Constant 节点
used_inputs = set()
for node in graph.node:
    used_inputs.update(node.input)

filtered_nodes = []
for node in graph.node:
    if node.op_type == "Constant" and all(o not in used_inputs for o in node.output):
        continue
    filtered_nodes.append(node)

del graph.node[:]
graph.node.extend(filtered_nodes)

model.opset_import.extend([helper.make_opsetid("my.custom", 1)])
onnx.save(model, "work/mini_custom.onnx")
print("saved: work/mini_custom.onnx")

saved: work/mini_custom.onnx


我们规定：

MyScale(x, alpha) = x * alpha

所以整个模型变成：

output = MyScale(Relu(x), alpha=2.0)

如果输入：

[-1.0, 0.5, 2.0]

那么：

Relu -> [0.0, 0.5, 2.0]
MyScale -> [0.0, 1.0, 4.0]

开始op-package写 OpDef XML
写 OpDef XML

你贴的文档已经说明了，OpDef 里至少要有：

Name
Input
Output
Parameter（可选）
SupportedBackend
UseDefaultTranslation

并且 custom op 至少要有一个输入和一个输出

这里的关键点：

Reference Source="ONNX"：给 converter 做 source-side 对应
UseDefaultTranslation=false：表示这是 generic custom op，不是覆盖 QNN 原生 op
SupportedBackend=CPU：先走最小闭环，不碰 HTP

In [ ]:
<OpDefCollection PackageName="MyScaleOpPackage">

  <OpDef>
    <Name>MyScale</Name>

    <Description>
      <Content>Multiply input tensor by scalar alpha</Content>
    </Description>

    <Reference Source="ONNX" Url="custom://my.custom/MyScale"></Reference>

    <Input>
      <Name>input</Name>
      <Mandatory>true</Mandatory>
      <Datatype>QNN_DATATYPE_FLOAT_32</Datatype>
      <Shape>
        <Rank>ND</Rank>
        <Layout>UNDEFINED</Layout>
      </Shape>
    </Input>

    <Output>
      <Name>output</Name>
      <Mandatory>true</Mandatory>
      <Datatype>QNN_DATATYPE_FLOAT_32</Datatype>
      <Shape>
        <Rank>ND</Rank>
        <Layout>UNDEFINED</Layout>
      </Shape>
    </Output>

    <Parameter>
      <Name>alpha</Name>
      <Mandatory>true</Mandatory>
      <Datatype>QNN_DATATYPE_FLOAT_32</Datatype>
      <Shape>
        <Rank>SCALAR</Rank>
        <Layout>UNDEFINED</Layout>
      </Shape>
      <Default>1.0</Default>
    </Parameter>

    <UseDefaultTranslation>false</UseDefaultTranslation>
    <SupportedBackend>CPU</SupportedBackend>
  </OpDef>

</OpDefCollection>

生成 custom op package 骨架

QNN/QAIRT 文档里有 qnn-op-package-generator 的说明，用来根据 op package 配置生成包骨架。

In [12]:
import subprocess

subprocess.run([
    "qnn-op-package-generator",
    "-p", "custom_op/custom_Op.xml",
    "--output_path", "work/op_package"
], check=True)

2026-04-05 11:17:21,023 - 278 - INFO - Note: Interface provider function will be named: MyScaleOpPackageInterfaceProvider
2026-04-05 11:17:21,035 - 278 - INFO - Code generation is complete for package: MyScaleOpPackage at /workspace/work/op_package


CompletedProcess(args=['qnn-op-package-generator', '-p', 'custom_op/custom_Op.xml', '--output_path', 'work/op_package'], returncode=0)

修改src下的ops，实现
所以实现一个op 我要关注的就是 权重怎么读进来 输入怎么读取 输出怎么输出 这几个主要的 就够了把
基本理解是对的，这三件是核心主线，但还需要再补几项“容易踩坑但必须做”的点，才算一个可用的 op。
最小完整清单建议是：

参数/权重读取
包括参数名、类型、shape、默认值、是否必填。

输入读取
校验输入个数、dtype、维度、layout、量化信息（如果有）。

输出写回
输出 shape 推导、dtype 一致性、内存可写性、边界检查。

配置校验
在创建或 finalize 阶段把不合法配置尽早拦掉，避免 execute 才崩。

生命周期管理
初始化阶段缓存元数据，执行阶段只做计算；释放阶段把申请的资源清干净。

错误码和健壮性
空指针、长度不匹配、类型不匹配都要返回明确错误码，不要静默失败。

性能与一致性
避免在 execute 里重复做昂贵解析；多线程/重复调用场景行为一致。

In [13]:
import subprocess

subprocess.run([
    "make",
    "-C", "work/op_package/MyScaleOpPackage",
    "cpu_x86"
], check=True)

make: Entering directory '/workspace/work/op_package/MyScaleOpPackage'
make -f makefiles/Makefile.linux-x86_64
make[1]: Entering directory '/workspace/work/op_package/MyScaleOpPackage'
Copying custom op source files from SDK
mkdir -p obj/x86_64-linux-clang
clang++ -std=c++11 -fno-exceptions -fPIC -pg -I/opt/qairt/2.44.0.260225/include/QNN -I include -I/opt/qairt/2.44.0.260225/include/QNN/CPU -I /opt/qairt/2.44.0.260225/share/QNN/OpPackageGenerator/CustomOp -I src/utils -I src/utils/CPU -march=x86-64 -O3 -Wno-write-strings -fvisibility=hidden -DQNN_API="__attribute__((visibility(\"default\")))" -c src/CpuCustomOpPackage.cpp -o obj/x86_64-linux-clang/CpuCustomOpPackage.o
clang++ -std=c++11 -fno-exceptions -fPIC -pg -I/opt/qairt/2.44.0.260225/include/QNN -I include -I/opt/qairt/2.44.0.260225/include/QNN/CPU -I /opt/qairt/2.44.0.260225/share/QNN/OpPackageGenerator/CustomOp -I src/utils -I src/utils/CPU -march=x86-64 -O3 -Wno-write-strings -fvisibility=hidden -DQNN_API="__attribute__((visib

src/ops/MyScale.cpp:26:62: error: no member named 'getUserData' in 'qnn::custom::utils::CustomOp'
  auto* opData = reinterpret_cast<MyScaleOpData*>(operation->getUserData());
                                                  ~~~~~~~~~  ^
src/ops/MyScale.cpp:30:32: error: no member named 'input' in 'qnn::custom::utils::CustomOp'
  auto& inTensor  = operation->input(0);
                    ~~~~~~~~~  ^
src/ops/MyScale.cpp:31:32: error: no member named 'output' in 'qnn::custom::utils::CustomOp'
  auto& outTensor = operation->output(0);
                    ~~~~~~~~~  ^
src/ops/MyScale.cpp:65:32: error: no member named 'input' in 'qnn::custom::utils::CustomOp'
  auto& inTensor  = operation->input(0);
                    ~~~~~~~~~  ^
src/ops/MyScale.cpp:66:32: error: no member named 'output' in 'qnn::custom::utils::CustomOp'
  auto& outTensor = operation->output(0);
                    ~~~~~~~~~  ^
src/ops/MyScale.cpp:75:35: error: no member named 'hasParam' in 'qnn::custom::utils::CustomOp'

CompletedProcess(args=['make', '-C', 'work/op_package/MyScaleOpPackage', 'cpu_x86'], returncode=0)